#Read Bronze

In [0]:
df_subjects = spark.read.format("delta").table("bronze.subjects")
df_classes = spark.read.format("delta").table("bronze.classes")
df_students = spark.read.format("delta").table("bronze.students")
df_teacher = spark.read.format("delta").table("bronze.teacher")
df_grades = spark.read.format("delta").table("bronze.grades")

# Adds time and file name as columns

In [0]:
from pyspark.sql.functions import current_timestamp, lit


df_teacher = df_teacher.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("teacher"))
df_classes = df_classes.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("classes"))
df_students = df_students.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("students"))
df_subjects = df_subjects.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("subjects"))
df_grades = df_grades.withColumn("data_hora_silver", current_timestamp()).withColumn("nome_tabela", lit("grades"))


#Created Helper Functions

In [0]:
def apply_name_rules(colname: str) -> str:
    n = colname.upper()
    n = n.replace("ID_", "IDENTIFIER_")
    n = n.replace("ID", "IDENTIFIER")
    return n

def safe_drop(df, cols):
    existing = set(df.columns)
    to_drop = [c for c in cols if c in existing]
    return df.drop(*to_drop) if to_drop else df

# Modify and Save Silver

In [0]:
dataframes = [df_teacher, df_classes, df_students, df_subjects, df_grades]

for currDf in dataframes:
    new_cols = [_apply_name_rules(c) for c in currDf.columns]
    currDf = currDf.toDF(*new_cols)

    currDf = _safe_drop(currDf, ["DATA_HORA_BRONZE", "NOME_ARQUIVO"])

    currDf.write.format('delta').mode("overwrite").saveAsTable("silver."+currDf.select("NOME_TABELA").first()["NOME_TABELA"])


## (SQL) SHOW TABLES IN silver

In [0]:
%sql
SHOW TABLES IN silver

database,tableName,isTemporary
silver,classes,false
silver,grades,false
silver,students,false
silver,subjects,false
silver,teacher,false
